# CrashVision — Re-entrenamiento M1: Más Épocas
### Sistema de Detección Automática de Accidentes de Tránsito

**Proyecto:** CrashVision — Inteligencia Artificial 1  
**Universidad:** UDLA (Universidad de las Américas, Quito)  
**Docente:** Prof. Enrique Vinicio Carrera  
**Autores:** Sebastian Abad · Daniel Cornejo  
**Objetivo M1:** Re-entrenar YOLOv8n y YOLOv8s con 150 epochs y `patience=20`  

---

## Contexto y motivación

En el sprint anterior ambos modelos completaron las 50 epochs **sin activar el early stopping**  
(`patience=10`). Eso indica que las curvas de loss seguían descendiendo al terminar — los modelos  
no habían convergido. El recall sobre videos reales tiene un techo de **75% (6/8)** y hay falsos  
positivos persistentes. M1 ataca directamente ese cuello de botella con el menor riesgo:  
simplemente dejar que el entrenamiento continúe.

| Parámetro | Entrenamiento original | M1 (este notebook) |
|---|---|---|
| `epochs` | 50 | **150** |
| `patience` | 10 | **20** |
| `batch` | 16 | 16 (sin cambio) |
| `imgsz` | 640 | 640 (sin cambio) |
| Dataset | Roboflow v2 | Roboflow v2 (sin cambio) |

---

## Contenido del notebook

| Sección | Descripción |
|---|---|
| 0 | Verificación del entorno (GPU, CUDA, RAM) |
| 1 | Montaje de Google Drive |
| 2 | Instalación de dependencias |
| 3 | Descarga del dataset Roboflow |
| 4 | Re-entrenamiento YOLOv8n M1 (150 epochs) |
| 5 | Re-entrenamiento YOLOv8s M1 (150 epochs) |
| 6 | Evaluación comparativa M1 vs. baseline en test set |
| 7 | Visualización de curvas de entrenamiento |
| 8 | Exportación de modelos a Google Drive |
| 9 | Resumen final y decisión |


---
## Sección 0 — Verificación del entorno


In [2]:
import os
# Forzar GPU 0 antes de importar torch — evita que VS Code/Jupyter inyecte CUDA_VISIBLE_DEVICES=""
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import subprocess, sys
import torch

print("=" * 55)
print("VERIFICACIÓN DE ENTORNO — CrashVision M1 (Local)")
print("=" * 55)

# GPU
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        encoding='utf-8').strip()
    print(f"\nGPU detectada:")
    for line in gpu_info.split('\n'):
        print(f" {line}")
except FileNotFoundError:
    print("\nGPU (nvidia-smi) NO detectada.")

# RAM
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM disponible: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB total")

# Python
print(f"\nPython: {sys.version.split()[0]}")

# CUDA via PyTorch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM total: {props.total_memory / 1e9:.1f} GB")
else:
    print("ADVERTENCIA: CUDA no detectado. Revisa el kernel del notebook.")

print("\n" + "=" * 55)

VERIFICACIÓN DE ENTORNO — CrashVision M1 (Local)

GPU detectada:
 NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB, 610.47

RAM disponible: 17.0 GB / 34.0 GB total

Python: 3.11.9
PyTorch: 2.11.0+cu128
CUDA disponible: True
Dispositivo CUDA: NVIDIA GeForce RTX 4060 Laptop GPU
   VRAM total: 8.6 GB



---
## Sección 1 — Configuración de carpetas locales

> Los modelos M1 se guardan en subcarpetas separadas (`models/m1/`)  
> para no sobreescribir los baselines originales (`best_n.pt`, `best_s.pt`).

In [3]:
import os
from pathlib import Path

# Carpeta raíz del proyecto (misma ubicación que este notebook)
BASE_DIR          = Path(r"C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x")
DRIVE_BASE        = str(BASE_DIR / "CrashVision")
DRIVE_MODELS_BASE = str(BASE_DIR / "CrashVision" / "models")
DRIVE_MODELS_M1   = str(BASE_DIR / "CrashVision" / "models" / "m1")
DRIVE_RESULTS_N   = str(BASE_DIR / "CrashVision" / "results" / "yolov8n_m1")
DRIVE_RESULTS_S   = str(BASE_DIR / "CrashVision" / "results" / "yolov8s_m1")
DRIVE_COMPARE_M1  = str(BASE_DIR / "CrashVision" / "results" / "comparison_m1")

for folder in [DRIVE_MODELS_M1, DRIVE_RESULTS_N, DRIVE_RESULTS_S, DRIVE_COMPARE_M1]:
    os.makedirs(folder, exist_ok=True)
    print(f"Carpeta lista: {folder}")

# Verificar que los baselines originales existen
for fname in ['best_n.pt', 'best_s.pt']:
    path = os.path.join(DRIVE_MODELS_BASE, fname)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"Baseline encontrado: {fname} ({size_mb:.1f} MB)")
    else:
        print(f"Baseline NO encontrado: {path}")
        print("La Sección 6 necesita estos archivos para la comparación.")

print("\nEstructura local:")
print(f"  {DRIVE_BASE}/")
print("  ├── models/")
print("  │   ├── best_n.pt        ← baseline original (NO se sobreescribe)")
print("  │   ├── best_s.pt        ← baseline original (NO se sobreescribe)")
print("  │   └── m1/")
print("  │       ├── best_n_m1.pt ← modelo n re-entrenado")
print("  │       └── best_s_m1.pt ← modelo s re-entrenado")
print("  └── results/")
print("      ├── yolov8n_m1/      ← curvas y métricas n M1")
print("      ├── yolov8s_m1/      ← curvas y métricas s M1")
print("      └── comparison_m1/   ← tabla comparativa M1 vs baseline")

Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m1
Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8n_m1
Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8s_m1
Carpeta lista: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\comparison_m1
Baseline NO encontrado: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\best_n.pt
La Sección 6 necesita estos archivos para la comparación.
Baseline NO encontrado: C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\best_s.pt
La Sección 6 necesita estos archivos para la comparación.

Estructura local:
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision/
  ├── models/
  │   ├── best_n.pt        ← baseline original (NO se sobreescribe)
  │   ├── best_s.pt        ← baseline original (NO se sobreescribe)
  │   └── m1/
  │       ├── best_n_m1.pt ← modelo n 

---
## Sección 2 — Instalación de dependencias


In [4]:
import ultralytics
import roboflow
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Roboflow: {roboflow.__version__}")

from ultralytics import YOLO
print("YOLOv8 importado correctamente")

import os, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
print("Librerías auxiliares listas")

Ultralytics: 8.4.75
Roboflow: 1.3.10
YOLOv8 importado correctamente
Librerías auxiliares listas


---
## Sección 3 — Descarga del dataset Roboflow

> Mismo dataset que el entrenamiento original: workspace `accident-detection-model`, versión 2.  
> La descarga tarda ~1–2 minutos la primera vez; Colab la cachea si ya existe.


In [5]:
from roboflow import Roboflow
from pathlib import Path
import yaml

ROBOFLOW_API_KEY = "DwCmQ00XOYIzR0pfSFUK"
WORKSPACE_SLUG = "accident-detection-model"
PROJECT_SLUG = "accident-detection-model"
DATASET_VERSION = 2

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_SLUG).project(PROJECT_SLUG)
dataset = project.version(DATASET_VERSION).download("yolov8")

DATASET_PATH = Path(dataset.location)
DATA_YAML    = str(DATASET_PATH / 'data.yaml')

print(f"\nDataset descargado en: {DATASET_PATH}")

# Validar data.yaml
with open(DATA_YAML, 'r') as f:
    yaml_content = yaml.safe_load(f)

names_lower = [n.lower() for n in yaml_content.get('names', [])]
assert yaml_content.get('nc') == 1, "Se esperaba nc=1"
assert 'accident' in names_lower, "Clase 'accident' no encontrada"
print(f"data.yaml válido — nc={yaml_content['nc']}, clase='{yaml_content['names'][0]}'")
print(f"data.yaml → {DATA_YAML}")


loading Roboflow workspace...
loading Roboflow project...

Dataset descargado en: c:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2
data.yaml válido — nc=1, clase='Accident'
data.yaml → c:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2\data.yaml


---
## Sección 4 — Re-entrenamiento YOLOv8n M1

### Cambios respecto al entrenamiento original

| Parámetro | Baseline | M1 | Razón |
|---|---|---|---|
| `epochs` | 50 | **150** | El modelo no convergió en 50 epochs (early stopping no activado) |
| `patience` | 10 | **20** | Más tolerancia para salir de mesetas antes de parar |
| Base | `yolov8n.pt` (COCO) | `yolov8n.pt` (COCO) | Re-entrenamiento desde cero para comparación limpia |

> ⏱️ **Tiempo estimado:** ~90–110 minutos en GPU T4 (si no hay early stopping antes de epoch 150)


In [6]:
from ultralytics import YOLO
import torch, time

# Hiperparámetros M1 - YOLOv8n
EPOCHS_M1    = 150
PATIENCE_M1  = 20
BATCH_SIZE   = 16
IMG_SIZE     = 640
DEVICE       = 0  # GPU — RTX 4060 Laptop
MODEL_NAME_N = 'crashvision_n_m1'

print("=" * 58)
print("RE-ENTRENAMIENTO YOLOv8n M1 - CrashVision")
print("=" * 58)
print(f"epochs={EPOCHS_M1} | patience={PATIENCE_M1} | batch={BATCH_SIZE} | imgsz={IMG_SIZE}")
print(f"device={DEVICE} | name={MODEL_NAME_N}")
print("=" * 58)

model_n_m1 = YOLO('yolov8n.pt')  # base COCO, entrenamiento desde cero

t_start_n = time.time()

results_n_m1 = model_n_m1.train(
    data=DATA_YAML,
    epochs=EPOCHS_M1,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE_M1,
    device=DEVICE,
    name=MODEL_NAME_N,
    exist_ok=True,
    workers=2,
    cache=False,
    save=True,
    plots=True,
    verbose=True,
)

t_elapsed_n = time.time() - t_start_n
print(f"\nYOLOv8n M1 completado en {t_elapsed_n/60:.1f} minutos")

csv_n = Path(f'runs/detect/{MODEL_NAME_N}/results.csv')
if csv_n.exists():
    df_n_check = pd.read_csv(str(csv_n))
    actual_epochs_n = len(df_n_check)
    if actual_epochs_n < EPOCHS_M1:
        print(f"Early stopping activado en epoch {actual_epochs_n} / {EPOCHS_M1}  ← modelo convergió")
    else:
        print(f"El modelo completó las {EPOCHS_M1} epochs SIN early stopping.")
        print("Considera subir epochs a 200 o revisar las curvas de loss.")

RE-ENTRENAMIENTO YOLOv8n M1 - CrashVision
epochs=150 | patience=20 | batch=16 | imgsz=640
device=0 | name=crashvision_n_m1
Ultralytics 8.4.75  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

---
## Sección 5 — Re-entrenamiento YOLOv8s M1

> Los hiperparámetros son idénticos al n M1 para mantener la comparación justa.  
> ⏱️ **Tiempo estimado:** ~160–200 minutos en GPU T4.
> Si Colab expira, la Sección 8 guarda los `.pt` en Drive al terminar cada modelo.


In [7]:
from ultralytics import YOLO
import torch, time

MODEL_NAME_S = 'crashvision_s_m1'

print("=" * 58)
print("  RE-ENTRENAMIENTO YOLOv8s M1 — CrashVision")
print("=" * 58)
print(f"  epochs={EPOCHS_M1} | patience={PATIENCE_M1} | batch={BATCH_SIZE} | imgsz={IMG_SIZE}")
print(f"  device={DEVICE} | name={MODEL_NAME_S}")
print("=" * 58)

model_s_m1 = YOLO('yolov8s.pt')  # base COCO, entrenamiento desde cero

t_start_s = time.time()

results_s_m1 = model_s_m1.train(
    data=DATA_YAML,
    epochs=EPOCHS_M1,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE_M1,
    device=DEVICE,
    name=MODEL_NAME_S,
    exist_ok=True,
    workers=2,
    cache=False,
    save=True,
    plots=True,
    verbose=True,
)

t_elapsed_s = time.time() - t_start_s
print(f"\nYOLOv8s M1 completado en {t_elapsed_s/60:.1f} minutos")

csv_s = Path(f'runs/detect/{MODEL_NAME_S}/results.csv')
if csv_s.exists():
    df_s_check = pd.read_csv(str(csv_s))
    actual_epochs_s = len(df_s_check)
    if actual_epochs_s < EPOCHS_M1:
        print(f"Early stopping activado en epoch {actual_epochs_s} / {EPOCHS_M1}  ← modelo convergió")
    else:
        print(f"El modelo completó las {EPOCHS_M1} epochs SIN early stopping.")
        print("Considera subir epochs a 200 o continuar con M2 (data augmentation).")

  RE-ENTRENAMIENTO YOLOv8s M1 — CrashVision
  epochs=150 | patience=20 | batch=16 | imgsz=640
  device=0 | name=crashvision_s_m1
Ultralytics 8.4.75  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4

---
## Sección 6 — Evaluación comparativa M1 vs. Baseline

Evaluamos los 4 modelos en el test set con `conf=0.30` (threshold óptimo del Experimento 1)  
para ver si M1 mejora las métricas respecto a los baselines originales.


In [8]:
from ultralytics import YOLO

CONF_EVAL = 0.30  # threshold óptimo - Experimento 1

# Rutas de los 4 modelos a comparar
models_eval = {
    'YOLOv8n Baseline': f'{DRIVE_MODELS_BASE}/best_n.pt',
    'YOLOv8s Baseline': f'{DRIVE_MODELS_BASE}/best_s.pt',
    'YOLOv8n M1':       f'runs/detect/{MODEL_NAME_N}/weights/best.pt',
    'YOLOv8s M1':       f'runs/detect/{MODEL_NAME_S}/weights/best.pt',
}

# Verificar que todos existen
for name, path in models_eval.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"{name:<22} → {size_mb:.1f} MB")
    else:
        print(f"NO encontrado: {name} → {path}")

print(f"\nEvaluando en test set (conf={CONF_EVAL})...")
print("-" * 65)

comparison_results = []

for model_name, model_path in models_eval.items():
    if not os.path.exists(model_path):
        print(f"Saltando {model_name} (archivo no encontrado)")
        continue

    print(f"  Evaluando {model_name}...", end=' ', flush=True)
    m_obj = YOLO(model_path)
    m = m_obj.val(
        data=DATA_YAML,
        split='test',
        conf=CONF_EVAL,
        iou=0.50,
        verbose=False,
    )
    precision = m.box.mp
    recall    = m.box.mr
    map50     = m.box.map50
    map_full  = m.box.map
    f1        = 2 * (precision * recall) / (precision + recall + 1e-9)
    size_mb   = os.path.getsize(model_path) / 1e6

    comparison_results.append({
        'Modelo':     model_name,
        'Precision':  round(precision, 4),
        'Recall':     round(recall, 4),
        'F1-score':   round(f1, 4),
        'mAP@50':     round(map50, 4),
        'mAP@50:95':  round(map_full, 4),
        'Tamaño(MB)': round(size_mb, 1),
    })
    print(f"P={precision:.3f} | R={recall:.3f} | F1={f1:.3f} | mAP@50={map50:.3f}")

df_comparison = pd.DataFrame(comparison_results)

print("\n" + "=" * 75)
print("  COMPARACIÓN M1 vs BASELINE (conf=0.30, test set)")
print("=" * 75)
print(df_comparison.to_string(index=False))

# Análisis automático de mejora
print("\n── Análisis de mejora ──")
for base_name, m1_name in [('YOLOv8n Baseline', 'YOLOv8n M1'), ('YOLOv8s Baseline', 'YOLOv8s M1')]:
    base_rows = df_comparison[df_comparison['Modelo'] == base_name]
    m1_rows   = df_comparison[df_comparison['Modelo'] == m1_name]
    if base_rows.empty or m1_rows.empty:
        continue
    base_row = base_rows.iloc[0]
    m1_row   = m1_rows.iloc[0]
    delta_f1    = m1_row['F1-score'] - base_row['F1-score']
    delta_map50 = m1_row['mAP@50']   - base_row['mAP@50']
    delta_rec   = m1_row['Recall']   - base_row['Recall']
    sign = lambda v: f"+{v:.4f}" if v >= 0 else f"{v:.4f}"
    print(f"  {m1_name:<16}: ΔF1={sign(delta_f1)} | ΔmAP@50={sign(delta_map50)} | ΔRecall={sign(delta_rec)}")

# Guardar CSV
csv_out = f'{DRIVE_COMPARE_M1}/m1_vs_baseline.csv'
df_comparison.to_csv(csv_out, index=False)
print(f"\nm1_vs_baseline.csv guardado en Drive")


NO encontrado: YOLOv8n Baseline → C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models/best_n.pt
NO encontrado: YOLOv8s Baseline → C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models/best_s.pt
YOLOv8n M1             → 6.3 MB
YOLOv8s M1             → 22.5 MB

Evaluando en test set (conf=0.3)...
-----------------------------------------------------------------
Saltando YOLOv8n Baseline (archivo no encontrado)
Saltando YOLOv8s Baseline (archivo no encontrado)
  Evaluando YOLOv8n M1... Ultralytics 8.4.75  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 116.032.6 MB/s, size: 63.5 KB)
val: Scanning C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\Accident-detection-model-2\test\labels... 362 images, 186 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 362/362 1.1Kit/s 0.3s<0.1s
val: New cache cre

---
## Sección 7 — Visualización de curvas de entrenamiento

Comparamos las curvas de loss y mAP de M1 vs. el baseline para ver si la convergencia mejoró.


In [9]:
# helper
def safe_plot(ax, df, col_name, title, color, ylabel=''):
    """Grafica una columna si existe, con matching parcial de nombre."""
    match = [c for c in df.columns if col_name.lower() in c.lower()]
    if match:
        epoch_col = 'epoch' if 'epoch' in df.columns else df.columns[0]
        ax.plot(df[epoch_col], df[match[0]], color=color, linewidth=1.8)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel or title)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
    else:
        ax.text(0.5, 0.5, f'No disponible\n({col_name})', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(title, fontsize=10)


# Curvas YOLOv8n M1
csv_n_m1 = f'runs/detect/{MODEL_NAME_N}/results.csv'
if os.path.exists(csv_n_m1):
    df_n = pd.read_csv(csv_n_m1)
    df_n.columns = df_n.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'Curvas de Entrenamiento — YOLOv8n M1 ({len(df_n)} epochs)', fontsize=14, fontweight='bold')

    safe_plot(axes[0,0], df_n, 'train/box_loss',   'Box Loss (train)',    '#E74C3C', 'Loss')
    safe_plot(axes[0,1], df_n, 'train/cls_loss',   'Class Loss (train)',  '#E67E22', 'Loss')
    safe_plot(axes[0,2], df_n, 'train/dfl_loss',   'DFL Loss (train)',    '#F39C12', 'Loss')
    safe_plot(axes[1,0], df_n, 'metrics/mAP50',    'mAP@50 (val)',        '#27AE60', 'mAP')
    safe_plot(axes[1,1], df_n, 'metrics/precision','Precision (val)',      '#2980B9', 'Precision')
    safe_plot(axes[1,2], df_n, 'metrics/recall',   'Recall (val)',         '#8E44AD', 'Recall')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_RESULTS_N}/training_curves_m1.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Curvas YOLOv8n M1 guardadas en Drive")
else:
    print(f"results.csv no encontrado: {csv_n_m1}")


<Figure size 1600x900 with 6 Axes>

Curvas YOLOv8n M1 guardadas en Drive


In [10]:
# Curvas YOLOv8s M1
csv_s_m1 = f'runs/detect/{MODEL_NAME_S}/results.csv'
if os.path.exists(csv_s_m1):
    df_s = pd.read_csv(csv_s_m1)
    df_s.columns = df_s.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'Curvas de Entrenamiento - YOLOv8s M1 ({len(df_s)} epochs)', fontsize=14, fontweight='bold')

    safe_plot(axes[0,0], df_s, 'train/box_loss',   'Box Loss (train)',    '#E74C3C', 'Loss')
    safe_plot(axes[0,1], df_s, 'train/cls_loss',   'Class Loss (train)',  '#E67E22', 'Loss')
    safe_plot(axes[0,2], df_s, 'train/dfl_loss',   'DFL Loss (train)',    '#F39C12', 'Loss')
    safe_plot(axes[1,0], df_s, 'metrics/mAP50',    'mAP@50 (val)',        '#27AE60', 'mAP')
    safe_plot(axes[1,1], df_s, 'metrics/precision','Precision (val)',      '#2980B9', 'Precision')
    safe_plot(axes[1,2], df_s, 'metrics/recall',   'Recall (val)',         '#8E44AD', 'Recall')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_RESULTS_S}/training_curves_m1.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Curvas YOLOv8s M1 guardadas en Drive")
else:
    print(f"results.csv no encontrado: {csv_s_m1}")


<Figure size 1600x900 with 6 Axes>

Curvas YOLOv8s M1 guardadas en Drive


In [11]:
# Gráfica comparativa M1 vs Baseline
if len(df_comparison) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    fig.suptitle('M1 vs Baseline — CrashVision (conf=0.30, test set)', fontsize=14, fontweight='bold')

    model_labels = df_comparison['Modelo'].tolist()
    x = np.arange(len(model_labels))
    colors = ['#95A5A6', '#95A5A6', '#E74C3C', '#E74C3C']  # gris=baseline, rojo=M1
    # Ajustar si hay menos modelos
    colors = colors[:len(model_labels)]

    # Subplot 1: F1 y mAP@50
    ax1 = axes[0]
    w = 0.3
    f1_vals   = df_comparison['F1-score'].tolist()
    map50_vals = df_comparison['mAP@50'].tolist()
    b1 = ax1.bar(x - w/2, f1_vals,    w, label='F1-score', color='#2980B9', alpha=0.85)
    b2 = ax1.bar(x + w/2, map50_vals, w, label='mAP@50',   color='#27AE60', alpha=0.85)
    for bar, v in list(zip(b1, f1_vals)) + list(zip(b2, map50_vals)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax1.set_ylim(0, 1.05)
    ax1.set_title('F1-score y mAP@50')
    ax1.set_ylabel('Score')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Subplot 2: Precision y Recall
    ax2 = axes[1]
    prec_vals = df_comparison['Precision'].tolist()
    rec_vals  = df_comparison['Recall'].tolist()
    b3 = ax2.bar(x - w/2, prec_vals, w, label='Precision', color='#E67E22', alpha=0.85)
    b4 = ax2.bar(x + w/2, rec_vals,  w, label='Recall',    color='#8E44AD', alpha=0.85)
    for bar, v in list(zip(b3, prec_vals)) + list(zip(b4, rec_vals)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax2.set_xticks(x)
    ax2.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax2.set_ylim(0, 1.05)
    ax2.set_title('Precision y Recall')
    ax2.set_ylabel('Score')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

    # Subplot 3: mAP@50:95
    ax3 = axes[2]
    map_full_vals = df_comparison['mAP@50:95'].tolist()
    bars = ax3.bar(model_labels, map_full_vals, color=colors, alpha=0.85, width=0.5)
    for bar, v in zip(bars, map_full_vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=8)
    ax3.set_ylim(0, 0.7)
    ax3.set_title('mAP@50:95')
    ax3.set_ylabel('mAP@50:95')
    ax3.grid(axis='y', alpha=0.3)

    # Línea horizontal de referencia (baseline YOLOv8s mAP@50 = 0.8278)
    for ax in [ax1, ax2]:
        ax.axhline(y=0.8278, color='gray', linestyle='--', linewidth=1, alpha=0.6,
                   label='mAP@50 baseline s')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_COMPARE_M1}/m1_vs_baseline_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Gráfica comparativa guardada en Drive")


set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.


<Figure size 1600x600 with 3 Axes>

Gráfica comparativa guardada en Drive


---
## Sección 8 — Exportación de modelos a carpeta local

> Los modelos M1 se guardan en `models/m1/` para NO sobreescribir los baselines originales.  
> Si M1 supera al baseline, copia manualmente `best_s_m1.pt` sobre `best_s.pt`.

In [12]:
import shutil, os

exports = [
    (
        f'runs/detect/{MODEL_NAME_N}/weights/best.pt',
        os.path.join(DRIVE_MODELS_M1, 'best_n_m1.pt'),
        f'runs/detect/{MODEL_NAME_N}/weights/last.pt',
        os.path.join(DRIVE_MODELS_M1, 'last_n_m1.pt'),
        DRIVE_RESULTS_N,
        MODEL_NAME_N,
        'YOLOv8n M1',
    ),
    (
        f'runs/detect/{MODEL_NAME_S}/weights/best.pt',
        os.path.join(DRIVE_MODELS_M1, 'best_s_m1.pt'),
        f'runs/detect/{MODEL_NAME_S}/weights/last.pt',
        os.path.join(DRIVE_MODELS_M1, 'last_s_m1.pt'),
        DRIVE_RESULTS_S,
        MODEL_NAME_S,
        'YOLOv8s M1',
    ),
]

print("=" * 58)
print("EXPORTANDO MODELOS M1 A CARPETA LOCAL")
print("=" * 58)

for best_src, best_dst, last_src, last_dst, results_dst, model_name_local, label in exports:
    if not os.path.exists(best_src):
        print(f"{label}: best.pt no encontrado en {best_src}")
        continue

    shutil.copy(best_src, best_dst)
    if os.path.exists(last_src):
        shutil.copy(last_src, last_dst)

    size_mb = os.path.getsize(best_dst) / 1e6
    print(f"\n{label}")
    print(f"{best_dst} ({size_mb:.1f} MB)")

    results_dir = Path(f'runs/detect/{model_name_local}')
    artifacts   = [
        'results.csv',
        'confusion_matrix.png',
        'BoxPR_curve.png', 'PR_curve.png',
        'BoxF1_curve.png', 'F1_curve.png',
        'results.png',
    ]
    for fname in artifacts:
        src = results_dir / fname
        if src.exists():
            shutil.copy(str(src), os.path.join(results_dst, fname))
            print(f"  {fname}")

print("\n" + "=" * 58)
print("  Para reemplazar el modelo de producción si M1 es mejor:")
print(f"  shutil.copy('{DRIVE_MODELS_M1}/best_s_m1.pt', '{DRIVE_MODELS_BASE}/best_s.pt')")

EXPORTANDO MODELOS M1 A CARPETA LOCAL

YOLOv8n M1
C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m1\best_n_m1.pt (6.3 MB)
  results.csv
  confusion_matrix.png
  BoxPR_curve.png
  BoxF1_curve.png
  results.png

YOLOv8s M1
C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m1\best_s_m1.pt (22.5 MB)
  results.csv
  confusion_matrix.png
  BoxPR_curve.png
  BoxF1_curve.png
  results.png

  Para reemplazar el modelo de producción si M1 es mejor:
  shutil.copy('C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m1/best_s_m1.pt', 'C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models/best_s.pt')


---
## Sección 9 — Resumen final y decisión


In [13]:
print("\n" + "=" * 68)
print("  RESUMEN FINAL — CrashVision M1")
print("=" * 68)

print("\nEarly stopping:")
for csv_path, label in [(csv_n_m1, 'YOLOv8n M1'), (csv_s_m1, 'YOLOv8s M1')]:
    if os.path.exists(csv_path):
        df_tmp = pd.read_csv(csv_path)
        n_ep = len(df_tmp)
        status = f"paró en epoch {n_ep} (convergió)" if n_ep < EPOCHS_M1 else f"completó {n_ep} epochs sin parar"
        print(f"   {label:<18}: {status}")

if not df_comparison.empty:
    print("\nComparación M1 vs Baseline (conf=0.30, test set):")
    print(df_comparison[['Modelo','Precision','Recall','F1-score','mAP@50','mAP@50:95']].to_string(index=False))

print("\nDecisión de producción:")
s_base_rows = df_comparison[df_comparison['Modelo'] == 'YOLOv8s Baseline']
s_m1_rows   = df_comparison[df_comparison['Modelo'] == 'YOLOv8s M1']
if not s_base_rows.empty and not s_m1_rows.empty:
    s_base = s_base_rows.iloc[0]
    s_m1   = s_m1_rows.iloc[0]
    if s_m1['F1-score'] > s_base['F1-score']:
        delta = s_m1['F1-score'] - s_base['F1-score']
        print(f"YOLOv8s M1 SUPERA al baseline en F1 por +{delta:.4f}")
        print(f" → Copiar {DRIVE_MODELS_M1}/best_s_m1.pt sobre {DRIVE_MODELS_BASE}/best_s.pt")
        print(f" → Actualizar Config.TRAINING_METRICS en config.py con los nuevos valores")
    else:
        delta = s_base['F1-score'] - s_m1['F1-score']
        print(f"YOLOv8s M1 NO supera al baseline (ΔF1={-delta:.4f})")
        print(f" → Mantener best_s.pt original")
        print(f" → Continuar con M2 (data augmentation dirigido)")

print("\nArchivos generados localmente:")
local_files = [
    os.path.join(DRIVE_MODELS_M1, 'best_n_m1.pt'),
    os.path.join(DRIVE_MODELS_M1, 'best_s_m1.pt'),
    os.path.join(DRIVE_RESULTS_N, 'results.csv'),
    os.path.join(DRIVE_RESULTS_N, 'training_curves_m1.png'),
    os.path.join(DRIVE_RESULTS_S, 'results.csv'),
    os.path.join(DRIVE_RESULTS_S, 'training_curves_m1.png'),
    os.path.join(DRIVE_COMPARE_M1, 'm1_vs_baseline.csv'),
    os.path.join(DRIVE_COMPARE_M1, 'm1_vs_baseline_comparison.png'),
]
for f in local_files:
    print(f"  {f}")

print("\nPróximos pasos según resultado:")
print("Si M1 mejoró → reemplazar best_s.pt + actualizar config.py")
print("Si M1 NO mejoró → continuar con M2 (data augmentation) o M3 (dataset + YOLOv8m)")
print("\n" + "=" * 68)


  RESUMEN FINAL — CrashVision M1

Early stopping:
   YOLOv8n M1        : paró en epoch 69 (convergió)
   YOLOv8s M1        : paró en epoch 92 (convergió)

Comparación M1 vs Baseline (conf=0.30, test set):
    Modelo  Precision  Recall  F1-score  mAP@50  mAP@50:95
YOLOv8n M1     0.8687  0.7353    0.7964  0.7017     0.3503
YOLOv8s M1     0.8692  0.8775    0.8733  0.8576     0.4353

Decisión de producción:

Archivos generados localmente:
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m1\best_n_m1.pt
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\models\m1\best_s_m1.pt
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8n_m1\results.csv
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8n_m1\training_curves_m1.png
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8s_m1\results.csv
  C:\Users\dddma\OneDrive\Escritorio\Codigos_repors\x\CrashVision\results\yolov8